# Validation du modèle "Rolling Window"

In [10]:
%load_ext autoreload
%autoreload 2

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset 

# Tes modules
from ml_in_finance_ensae.data import DataPaths, load_all_test_assets_excess, load_macro_fred # Tes fonctions d'import
from ml_in_finance_ensae.data_utils import AssetPricingDataset, apply_stationarity_transforms, add_group_stats
from ml_in_finance_ensae.networks import SDFNetwork, MacroLSTM, AdversarialNetwork
from ml_in_finance_ensae.gan_engine import GANTrainer, GANTrainerEnsemble
from ml_in_finance_ensae.metrics import evaluate_performance, calculate_sharpe_ratio, calculate_r2_scores, evaluate_performance_ensemble

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
# A. Import macro et stationnarisation
raw_macro, tcodes = load_macro_fred('data/raw/2026-03-MD.csv')

returns_raw = load_all_test_assets_excess(DataPaths())
returns_num = returns_raw.apply(pd.to_numeric, errors='coerce')
returns_targeted = returns_num.loc['1959-01-01':]
returns = returns_targeted.dropna(axis=1)
returns = returns / 100.0
# D. Import des Rendements (FF25 + Ind49 = 74 actifs)

cols_to_drop = ['ACOGNO', 'ANDENOx', 'TWEXAFEGSMTHx']
raw_macro = raw_macro.drop(columns=cols_to_drop, errors='ignore')
raw_macro = raw_macro.dropna()

# Appliquer tcode individuel d'abord
df_macro = apply_stationarity_transforms(raw_macro, tcodes) 
df_macro = add_group_stats(df_macro) 


# Aligner les dates entre Macro et Returns
common_idx = returns.index.intersection(df_macro.index)
returns = returns.loc[common_idx]
df_macro = df_macro.loc[common_idx]

/Users/daniel/Desktop/ENSAE/Cours 3A/ML_in_Finance_ENSAE/src/ml_in_finance_ensae/data_utils.py:62: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_transformed[col] = transforms[code](data[col])
/Users/daniel/Desktop/ENSAE/Cours 3A/ML_in_Finance_ENSAE/src/ml_in_finance_ensae/data_utils.py:62: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_transformed[col] = transforms[code](data[col])
/Users/daniel/Desktop/ENSAE/Cours 3A/ML_in_Finance_ENSAE/src/ml_in_finance_ensae/data_utils.py:62: PerformanceWarning: DataFrame is highly fragm

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_rolling_windows(returns, df_macro, train_size_years=30, test_size_years=1):
    train_sz = train_size_years * 12
    test_sz = test_size_years * 12
    total_months = len(returns)
    
    # Listes pour stocker les résultats de TOUTES les fenêtres de test
    all_oos_returns = []
    all_oos_omega = []
    all_oos_realized = []

    for start_idx in range(0, total_months - train_sz - test_sz, test_sz):
        train_end = start_idx + train_sz
        test_end = train_end + test_sz
        # 1. Split temporel
        train_ret = returns.iloc[start_idx:train_end]
        test_ret = returns.iloc[train_end:test_end]

        # POUR LA MACRO : On doit inclure le lookback pour le test !
        # Le test_macro doit commencer 'lookback' mois AVANT le début du test réel
        lookback = 12 
        train_macro_raw = df_macro.iloc[start_idx:train_end]

        # On prend la macro du test + les mois de lookback juste avant
        test_macro_raw = df_macro.iloc[train_end - lookback : test_end] 

        # 2. Normalisation (toujours sur les stats du train uniquement !)
        m_mean = train_macro_raw.mean()
        m_std = train_macro_raw.std().apply(lambda x: x if x > 1e-6 else 1.0) 

        train_macro = (train_macro_raw - m_mean) / m_std
        test_macro = (test_macro_raw - m_mean) / m_std 

        # 3. Création des Datasets
        # Vérifiez que len(test_ret) est bien > 0
        if len(test_ret) == 0:
            break # On a atteint la fin des données

        train_ds = AssetPricingDataset(train_ret, train_macro, lookback=lookback)
        test_ds = AssetPricingDataset(test_ret, test_macro, lookback=lookback)
        
        # 3. Ré-initialisation de l'Ensemble (Architecture CPZ)
        # On définit les dimensions ici
        N_ASSETS = returns.shape[1]      # 84
        N_MACRO = df_macro.shape[1]      # ~140
        HIDDEN_MACRO = 32               # Taille de l'état caché LSTM
        N_INSTRUMENTS = 8               # Nombre de fonctions g générées

        # On instancie les modèles de base
        lstm_base = MacroLSTM(input_dim=N_MACRO, lstm_hidden_dim=HIDDEN_MACRO).to(device)
        sdf_base = SDFNetwork(char_dim=N_ASSETS, macro_dim=HIDDEN_MACRO).to(device)
        adv_base = AdversarialNetwork(char_dim=N_ASSETS, macro_dim=HIDDEN_MACRO, output_dim=N_INSTRUMENTS).to(device)

        # On utilise le GANTrainerEnsemble pour stabiliser cette fenêtre
        model_trainer = GANTrainerEnsemble(
            sdf_base, adv_base, lstm_base, 
            lr_sdf=1e-4, lr_adv=1e-3, 
            n_models=3 # 3 modèles suffisent en rolling pour la stabilité
        )
        
        # 4. Préparation des DataLoaders
        test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)
        train_loader = DataLoader(train_ds, batch_size=1, shuffle=False)

        # 5. Entraînement court (30 époques max en rolling pour éviter l'overfitting)
        for epoch in range(30):
            for char, macro, ret in train_loader:
                model_trainer.train_step(char.to(device), macro.to(device), ret.to(device))
        
        # 6. Évaluation Hors-Échantillon (OOS) sur la fenêtre de test
        # On utilise la fonction d'évaluation qui moyenne l'ensemble
        metrics = evaluate_performance_ensemble(model_trainer, test_loader, device)
        
        # On accumule les résultats pour le calcul final global
        all_oos_returns.extend(metrics['sdf_returns_list'])
        all_oos_omega.append(metrics['all_omega'])
        all_oos_realized.append(metrics['all_returns'])
        
        print(f"Fenêtre finissant en {train_macro_raw.index[-1].year} terminée. Sharpe OOS: {metrics['Sharpe_Ratio']:.4f}")

    # 7. Calcul des métriques finales sur l'ensemble de la période OOS
    final_sharpe = calculate_sharpe_ratio(all_oos_returns)
    # Concaténation des listes de matrices pour le R2
    final_r2_total, _ = calculate_r2_scores(np.concatenate(all_oos_realized), np.concatenate(all_oos_omega))

    # Sauvegarde des rendements du portefeuille SDF (1 chiffre par mois)
    df_oos_ret = pd.DataFrame(all_oos_returns, columns=['sdf_return'])
    df_oos_ret.to_csv('all_oos_returns.csv', index=False)
    realized_array = np.concatenate(all_oos_realized, axis=0) 
    omega_array = np.concatenate(all_oos_omega, axis=0)

    # On sauve tout dans un seul fichier binaire
    np.savez('simulation_data_full.npz', realized=realized_array, omega=omega_array)

    print("✅ Données matricielles sauvegardées dans 'simulation_data_full.npz'")

    with open("performances_finales.txt", "w") as f:
        f.write("--- PERFORMANCE FINALE ROLLING WINDOW ---\n")
        f.write(f"Date de la simulation : {pd.Timestamp.now()}\n")
        f.write(f"Sharpe Global : {final_sharpe:.4f}\n")
        f.write(f"R2 Global : {final_r2_total:.4f}\n")
        f.write(f"Configuration : 3 modeles ensemble, train=30y, test=1y\n")

    print("✅ Rapport texte sauvegardé.")
        
    print(f"\n--- PERFORMANCE FINALE ROLLING WINDOW ---")
    print(f"Sharpe Global: {final_sharpe:.4f}")
    print(f"R2 Global: {final_r2_total:.4f}")


In [13]:
train_rolling_windows(returns, df_macro, train_size_years=30, test_size_years=1)

Fenêtre finissant en 1992 terminée. Sharpe OOS: -2.3830
Fenêtre finissant en 1993 terminée. Sharpe OOS: 0.3352
Fenêtre finissant en 1994 terminée. Sharpe OOS: -2.1592
Fenêtre finissant en 1995 terminée. Sharpe OOS: -0.5033
Fenêtre finissant en 1996 terminée. Sharpe OOS: -1.5444
Fenêtre finissant en 1997 terminée. Sharpe OOS: 1.8343
Fenêtre finissant en 1998 terminée. Sharpe OOS: 0.5212
Fenêtre finissant en 1999 terminée. Sharpe OOS: -0.8385
Fenêtre finissant en 2000 terminée. Sharpe OOS: 0.6959
Fenêtre finissant en 2001 terminée. Sharpe OOS: 1.9023
Fenêtre finissant en 2002 terminée. Sharpe OOS: -0.9853
Fenêtre finissant en 2003 terminée. Sharpe OOS: -1.2058
Fenêtre finissant en 2004 terminée. Sharpe OOS: -2.0590
Fenêtre finissant en 2005 terminée. Sharpe OOS: -0.7052
Fenêtre finissant en 2006 terminée. Sharpe OOS: 0.1515
Fenêtre finissant en 2007 terminée. Sharpe OOS: -0.9816
Fenêtre finissant en 2008 terminée. Sharpe OOS: -0.7608
Fenêtre finissant en 2009 terminée. Sharpe OOS: 1.1237

In [14]:
returns.head()

,FF25_SMALL LoBM,FF25_ME1 BM2,FF25_ME1 BM3,FF25_ME1 BM4,FF25_SMALL HiBM,FF25_ME2 BM1,FF25_ME2 BM2,FF25_ME2 BM3,FF25_ME2 BM4,FF25_ME2 BM5,...,MOM_Lo PRIOR,MOM_PRIOR 2,MOM_PRIOR 3,MOM_PRIOR 4,MOM_PRIOR 5,MOM_PRIOR 6,MOM_PRIOR 7,MOM_PRIOR 8,MOM_PRIOR 9,MOM_Hi PRIOR
Date,,,,,,,,,,,,,,,,,,,,,
1962-09-01,-0.000881,-0.000890,-0.001008,-0.000544,-0.000656,-0.000844,-0.000594,-0.000779,-0.000654,-0.000776,...,-0.000817,-0.000747,-0.000688,-0.000859,-0.000525,-0.000515,-0.000495,-0.000427,-0.000321,-0.000221
1962-10-01,-0.001063,-0.000765,-0.000374,-0.000560,-0.000261,-0.000505,-0.000556,-0.000486,-0.000157,-0.000104,...,-0.000169,0.000044,-0.000091,-0.000115,-0.000003,0.000171,0.000114,-0.000154,0.000100,-0.000051
1962-11-01,0.001450,0.001657,0.001197,0.001470,0.001352,0.001477,0.001554,0.001097,0.001451,0.001626,...,0.001748,0.001637,0.001496,0.001279,0.001212,0.001010,0.000860,0.000886,0.001019,0.000811
1962-12-01,-0.000359,-0.000508,-0.000087,-0.000253,-0.000242,-0.000315,-0.000111,-0.000120,-0.000233,-0.000386,...,-0.000199,-0.000283,-0.000116,-0.000016,-0.000172,0.000057,0.000233,0.000213,0.000168,0.000443
1963-01-01,0.001277,0.001119,0.000994,0.001050,0.001077,0.000737,0.000547,0.000714,0.000806,0.001082,...,0.000463,0.000766,0.000654,0.000609,0.000378,0.000558,0.000441,0.000481,0.000496,0.000292
